# Lecture 1 — OpenRouter API Testing

**Goal**: Explore different LLM models through OpenRouter API using resume data:

1. Check account credit balance
2. List available models with pricing
3. Compare outputs from Claude vs free/cheap models on resume analysis
4. Understand API request/response patterns
5. Observe differences in model capabilities

We'll use a dataset of 130 candidate resumes throughout this lecture series. Today we focus on API mechanics — using resumes as our test data to compare how different models handle the same tasks.

## Setup
This notebook requires:
- `OPENROUTER_API_KEY` (enter directly in Cell 1)


In [ ]:
# Add current directory to Python path for imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))


import json
from typing import Any, Dict, List
import pandas as pd

from openrouter_utils import (
    check_credits,
    print_remaining_credits,
    list_models,
    chat_completion,
    safe_chat,
    display_comparison,
    load_resumes,
    load_job_requirements
)

OPENROUTER_API_KEY = ""  # Paste your key here

# Models to test
MODELS = {
    "claude": "anthropic/claude-sonnet-4.6",
    "qwen-free": "qwen/qwen3.6-plus-preview:free",
}

print("Imports loaded")
print(f"Testing {len(MODELS)} models:", list(MODELS.keys()))

if not OPENROUTER_API_KEY or OPENROUTER_API_KEY.strip() == "":
    raise RuntimeError(
        "⚠️  Please set OPENROUTER_API_KEY above before running this notebook.\n"
        "Get your key from: https://openrouter.ai/keys"
    )

print("✓ API key configured")

In [ ]:
print_remaining_credits(OPENROUTER_API_KEY)

In [ ]:
# Execute - function is imported from openrouter_utils.py
models_list = list_models(OPENROUTER_API_KEY, limit=100)

# Parse into DataFrame for easy viewing
models_df = pd.DataFrame([
    {
        "id": m.get("id", ""),
        "name": m.get("name", ""),
        "prompt_cost": m.get("pricing", {}).get("prompt", "N/A"),
        "completion_cost": m.get("pricing", {}).get("completion", "N/A"),
        "context_length": m.get("context_length", "N/A"),
    }
    for m in models_list
])

print(f"Found {len(models_df)} models")
print("\nOur test models:")
for key, model_id in MODELS.items():
    match = models_df[models_df["id"] == model_id]
    if not match.empty:
        row = match.iloc[0]
        print(f"  {key:10s} → ${row['prompt_cost']}/1M prompt tokens")
    else:
        print(f"  {key:10s} → {model_id} (not found in list)")

# Display sample of all models
models_df.head(20)

In [ ]:
# Load resume data
resumes = load_resumes('../data/resumes_final.csv')
print(f"Loaded {len(resumes)} resumes")

# Pick one sample resume for our API tests
sample_id = list(resumes.keys())[0]
sample_resume = resumes[sample_id]['Resume_str']

print(f"\nSample resume ID: {sample_id}")
print(f"Preview (first 500 chars):\n{sample_resume[:500]}...")

# Define test prompts that showcase model differences using resume data
PROMPTS = [
    {
        "name": "Resume Summary",
        "messages": [
            {
                "role": "user",
                "content": f"Summarize this resume in 2-3 sentences:\n\n{sample_resume}"
            }
        ]
    },
    {
        "name": "Candidate Strengths",
        "messages": [
            {
                "role": "user",
                "content": f"What are this candidate's top 3 strengths? Be specific and cite evidence from the resume.\n\n{sample_resume}"
            }
        ]
    },
    {
        "name": "Experience Level",
        "messages": [
            {
                "role": "user",
                "content": f"Rate this candidate's experience level as one of: junior, mid, or senior. Explain your reasoning in 2-3 sentences.\n\n{sample_resume}"
            }
        ]
    }
]

print(f"\nDefined {len(PROMPTS)} test prompts: {[p['name'] for p in PROMPTS]}")

### Before Going Further

Verify the information above: 
* Where is the resume we are looking at?
* Did it get the correct information?

In [ ]:
# Run all prompts through all models

results = []

for prompt_obj in PROMPTS:
    prompt_name = prompt_obj["name"]
    messages = prompt_obj["messages"]
    
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt_name}")
    print(f"{'='*60}")
    
    for model_key, model_id in MODELS.items():
        print(f"\nTesting {model_key}...")
        
        result = chat_completion(
            OPENROUTER_API_KEY,
            model_id,
            messages,
            temperature=0.7,
            max_tokens=500
        )
        
        results.append({
            "prompt": prompt_name,
            "model_key": model_key,
            "model_id": model_id,
            **result
        })
        
        # Display output
        if result["error"]:
            print(f"  ❌ Error: {result['error']}")
        else:
            content = result["content"]
            preview = content
            print(f"  ✓ Response: {preview}")
            
            usage = result.get("usage", {})
            if usage:
                print(f"    Tokens: {usage.get('prompt_tokens', 0)} prompt + {usage.get('completion_tokens', 0)} completion")

print("\n✓ All tests complete")

### Before going further

* How do we add a different model? 
* Add 2-3 additional models to test the results

In [ ]:
# Convert results to DataFrame for analysis

results_df = pd.DataFrame(results)

# Add computed fields
results_df["response_length"] = results_df["content"].str.len()
results_df["has_error"] = results_df["error"].notna()
results_df["total_tokens"] = results_df["usage"].apply(
    lambda u: u.get("total_tokens", 0) if isinstance(u, dict) else 0
)

print(f"Collected {len(results_df)} responses")
print(f"Errors: {results_df['has_error'].sum()}")

# Summary by model
summary = results_df.groupby("model_key").agg({
    "has_error": "sum",
    "response_length": "mean",
    "total_tokens": "sum"
}).round(1)

summary.columns = ["Errors", "Avg Response Length", "Total Tokens Used"]
print("\nSummary by Model:")
print(summary)

results_df.head()

### Before Going Further

What are your take-aways from the table above?
* How doe the model resposnes compare? Why?

In [ ]:
# Function is now imported from openrouter_utils.py
# Display all comparisons
for prompt_obj in PROMPTS:
    display_comparison(results_df, prompt_obj["name"])

In [ ]:
# Structured Output Example: Extracting Resume Information
# This demonstrates extracting structured data from unstructured resume text

# Build the extraction prompt using the same sample resume
extraction_prompt = (
    "Extract key information from the following resume and return it as structured JSON.\n\n"
    f"Resume:\n{sample_resume}\n\n"
    "Extract the following information and return it as a JSON object with this exact structure:\n"
    "{\n"
    '  "candidate_name": "string or null if not found",\n'
    '  "current_title": "string (most recent job title)",\n'
    '  "years_of_experience": "number (estimate total years from work history)",\n'
    '  "education": {\n'
    '    "degree": "string (highest degree)",\n'
    '    "field": "string (field of study)",\n'
    '    "institution": "string (school name)"\n'
    "  },\n"
    '  "technical_skills": ["string (list of technical skills mentioned)"],\n'
    '  "soft_skills": ["string (list of soft/professional skills mentioned)"],\n'
    '  "experience_level": "junior | mid | senior (based on years and responsibilities)",\n'
    '  "industries": ["string (industries the candidate has worked in)"],\n'
    '  "certifications": ["string (any certifications mentioned)"],\n'
    '  "key_achievements": ["string (notable accomplishments, up to 3)"]\n'
    "}\n\n"
    "Important:\n"
    "- If a value is not mentioned in the resume, use null for that field\n"
    "- For lists, return an empty list [] if no items are found\n"
    "- Return ONLY valid JSON, no additional text or markdown formatting"
)

messages = [
    {"role": "user", "content": extraction_prompt}
]

output = {}
for model in MODELS:

    print(f"Running {model}")
    
    result = chat_completion(
        OPENROUTER_API_KEY,
        MODELS[model],
        messages,
        temperature=0.1,  # Very low temperature for consistent, accurate extraction
        max_tokens=800,   # Enough tokens for detailed JSON
        response_format={"type": "json_object"}  # Request JSON mode
    )

    output[model] = result

### Before going further

* What is a structured Response?
* Why will it help us?

In [ ]:
for model in output:
    if not output[model].get('parsed_content'):
        print(f"Model {model} has no structured output")
        continue
    
    parsed = output[model]['parsed_content']
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")
    print(f"  Name: {parsed.get('candidate_name', 'N/A')}")
    print(f"  Title: {parsed.get('current_title', 'N/A')}")
    print(f"  Experience Level: {parsed.get('experience_level', 'N/A')}")
    print(f"  Years of Experience: {parsed.get('years_of_experience', 'N/A')}")
    skills = parsed.get('technical_skills', [])
    print(f"  Technical Skills ({len(skills)}): {skills[:10]}")
    certs = parsed.get('certifications', [])
    print(f"  Certifications: {certs}")

## Preview: Job Requirements

In lectures 2-4, you'll use the job description below to **score and route candidates** against these requirements. For now, just read through it to get familiar with the role we'll be hiring for.


In [ ]:
# Load and display the job requirements we'll use in future lectures
from IPython.display import Markdown, display

job_req = load_job_requirements('../data/job_req_senior.md')
display(Markdown(job_req))